# Agrupamento hierárquico, caracterização e estabilidade

[▶ Abrir este notebook no Google Colab](https://colab.research.google.com/github/lalvim/disciplina_computacao_aplicada_humanidades_digitais/blob/main/unidade_09/02_hierarquia_caracterizacao_e_estabilidade.ipynb)

O agrupamento hierárquico aglomerativo começa com casos separados e os reúne. O
dendrograma mostra uma sequência de fusões, não uma taxonomia pronta. Cortá-lo em
três grupos é uma decisão analítica.

![Casos se fundem sucessivamente e um corte define a partição.](imagens/02_hierarquia.svg)

In [ ]:
# @title Preparação do ambiente — execute esta célula no Google Colab
from pathlib import Path
import importlib.util
import os
import subprocess
import sys

URL_REPOSITORIO = 'https://github.com/lalvim/disciplina_computacao_aplicada_humanidades_digitais.git'
REPOSITORIO = Path(
    "/content/disciplina_computacao_aplicada_humanidades_digitais"
)
PASTA_UNIDADE = REPOSITORIO / 'unidade_09'

try:
    import google.colab  # type: ignore  # noqa: F401
    EM_COLAB = True
except ImportError:
    EM_COLAB = False

if EM_COLAB:
    if not (REPOSITORIO / ".git").exists():
        subprocess.run(
            [
                "git",
                "clone",
                "--depth",
                "1",
                "--branch",
                "main",
                URL_REPOSITORIO,
                str(REPOSITORIO),
            ],
            check=True,
        )

    PACOTES_COLAB = [('sklearn', 'scikit-learn>=1.5,<2'), ('scipy', 'scipy>=1.11,<2')]
    ausentes = [
        especificacao
        for modulo, especificacao in PACOTES_COLAB
        if importlib.util.find_spec(modulo) is None
    ]
    if ausentes:
        subprocess.run(
            [sys.executable, "-m", "pip", "install", "-q", *ausentes],
            check=True,
        )

    os.chdir(PASTA_UNIDADE)
    print("Ambiente preparado em:", Path.cwd())
else:
    print("Ambiente local: nenhuma clonagem necessária.")

In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans, AgglomerativeClustering
from sklearn.metrics import adjusted_rand_score
from scipy.cluster.hierarchy import linkage, dendrogram
dados=pd.read_csv("dados/instituicoes_culturais.csv")
variaveis=["itens_acervo","eventos_ano","publico_ano","orcamento_anual"]
Xz=StandardScaler().fit_transform(dados[variaveis])
dados["cluster_hierarquico"]=AgglomerativeClustering(n_clusters=3,linkage="ward").fit_predict(Xz)
Z=linkage(Xz,method="ward")
plt.figure(figsize=(11,4)); dendrogram(Z,no_labels=True,color_threshold=None); plt.title("Dendrograma de Ward — dados fictícios"); plt.xlabel("Casos"); plt.ylabel("Distância de fusão"); plt.tight_layout(); plt.show()

## 2. Caracterizar sem batizar cedo demais

Médias descrevem perfis, mas nomes como “modernas” ou “precárias” acrescentariam
julgamento não contido nos dados. Comece por rótulos neutros e examine casos
próximos e distantes do centro antes de propor uma interpretação contextual.

In [ ]:
perfil=dados.groupby("cluster_hierarquico")[variaveis].agg(["mean","median","count"])
display(perfil.round(1))
display(pd.crosstab(dados["cluster_hierarquico"],dados["regiao"],normalize="index").round(2))

## 3. Estabilidade: a solução resiste a perturbações?

Repetimos K-means em subamostras de 80% e comparamos os rótulos dos casos comuns
pelo índice de Rand ajustado (ARI), que é invariável à troca dos números dos
clusters. A simulação não substitui validação externa, mas revela fragilidade.

In [ ]:
referencia=KMeans(n_clusters=3,random_state=42,n_init=20).fit_predict(Xz)
estabilidade=[]
rng=np.random.default_rng(42)
for repeticao in range(20):
    idx=np.sort(rng.choice(len(Xz),size=int(.8*len(Xz)),replace=False))
    parcial=KMeans(n_clusters=3,random_state=repeticao,n_init=20).fit_predict(Xz[idx])
    estabilidade.append(adjusted_rand_score(referencia[idx],parcial))
tabela_estabilidade=pd.Series(estabilidade,name="ARI").describe()
display(tabela_estabilidade.round(3))

### U09-A03 — Atividade — perfil, casos e estabilidade

Descreva os três grupos sem adjetivos valorativos; escolha dois casos de cada grupo
para inspeção; compare hierarquia e K-means; interprete mínimo, mediana e máximo do
ARI. **Dinâmica (35 minutos):** trios, com papéis de cálculo, leitura de casos e
crítica dos nomes. Proponham nomes somente ao final e marquem-nos como hipóteses.

**Resposta:** escreva aqui.

## Passagem

Variáveis numéricas tinham dimensões previamente definidas. Em textos, a própria
representação lexical decide quais semelhanças podem aparecer.